# Day 3: Building Pipelines with LCEL

**LangChain Hands-on Series | Day 3 of 10**
Trainer: Ajeetkumar

---

Yesterday you saw the pipe `|` a few times: `prompt | llm | parser`. Today we properly understand it, because almost every LangChain application you will ever build is made of these pipelines.

The official name is **LCEL** - LangChain Expression Language. Don't let the name scare you. It is just a way of connecting steps, like LEGO bricks that all have the same connector.

```
   input  -->  [ step A ]  -->  [ step B ]  -->  [ step C ]  -->  output
                 prompt            model           parser
```

Today's plan:

- why every piece in LangChain is a "Runnable", and why that matters
- the pipe `|` and what actually flows between the steps
- adding your own Python functions to a chain
- carrying data forward, running steps in parallel, and routing to different chains
- making chains survive failures (fallbacks and retries)
- **mini project:** a marketing content pipeline that writes and reviews its own post

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
parser = StrOutputParser()
print("Ready. Model:", MODEL_NAME)

## 1. Everything is a Runnable

Here is the single idea that makes LangChain click. Prompts, models, parsers, retrievers, tools, agents - they are all **Runnables**. A Runnable is anything with these methods:

| Method | Meaning |
|--------|---------|
| `invoke(x)` | run on one input |
| `batch([x1, x2])` | run on many inputs in parallel |
| `stream(x)` | give the output in pieces |
| `ainvoke`, `abatch`, `astream` | async versions (Day 9) |

Think of a universal power socket. Because every appliance has the same plug, you can plug anything into anything. Same here: because everything is a Runnable, anything can connect to anything.

Let's run three Runnables one by one and watch what each one hands to the next.

In [ ]:
prompt = ChatPromptTemplate.from_template("Tell me one fun fact about {animal} in one sentence.")

step1 = prompt.invoke({"animal": "octopus"})     # dict        -> PromptValue
step2 = llm.invoke(step1)                         # PromptValue -> AIMessage
step3 = parser.invoke(step2)                      # AIMessage   -> str

print(type(step1).__name__, "->", type(step2).__name__, "->", type(step3).__name__)
print(step3)

## 2. The pipe operator

That was three lines of "take the output, pass it on". The pipe does exactly that for us.

```
 {"animal": "octopus"}
         |
         v
   +-----------+      +---------+      +-----------------+
   |  prompt   | ---> |   llm   | ---> | StrOutputParser | --->  "Octopuses have three hearts."
   +-----------+      +---------+      +-----------------+
   dict->PromptValue  ->AIMessage          ->str
```

In [ ]:
fact_chain = prompt | llm | parser       # one Runnable made of three

print(fact_chain.invoke({"animal": "elephant"}))

**Pause and predict:** `fact_chain` is now a single Runnable. Does that mean it also has `.batch()` and `.stream()`, even though we never wrote them?

In [ ]:
for fact in fact_chain.batch([{"animal": "penguin"}, {"animal": "cheetah"}]):
    print("-", fact)

for piece in fact_chain.stream({"animal": "dolphin"}):
    print(piece, end="", flush=True)

<details>
<summary>Click to see the explanation</summary>

Yes. A chain of Runnables is itself a Runnable, so it gets `batch` and `stream` for free. Even streaming passes through the parser correctly - the parser forwards each text piece as it arrives. This is the payoff of the shared interface.
</details>

### See your chain as a picture

When chains get bigger, drawing them helps a lot. `get_graph().print_ascii()` prints the structure.

In [ ]:
fact_chain.get_graph().print_ascii()

## 3. Adding your own Python code to a chain

Not every step needs an LLM. Often you want to clean text, count something, or look up a database between LLM calls. `RunnableLambda` wraps any Python function so it can join a chain.

In [ ]:
from langchain_core.runnables import RunnableLambda

def word_count(text):
    return len(text.split())

count_words = RunnableLambda(word_count)     # now it has invoke, batch, stream...

print(count_words.invoke("LangChain makes LLM apps easy"))
print(count_words.batch(["one two", "one two three"]))

In [ ]:
fact_length_chain = fact_chain | count_words      # prompt -> llm -> parser -> our function

print("Words in the fact:", fact_length_chain.invoke({"animal": "giraffe"}))

Shortcut: if you put a plain function or lambda after a `|`, LangChain wraps it in `RunnableLambda` automatically.

In [ ]:
shout_chain = fact_chain | (lambda text: text.upper())
print(shout_chain.invoke({"animal": "koala"}))

## 4. Carrying data forward with RunnablePassthrough

Here is a problem you will hit very soon. In a chain, each step only sees the output of the step before. So after the LLM step, the original input is gone. What if a later step needs it?

`RunnablePassthrough.assign(...)` solves this. It takes the input dictionary, **keeps everything in it**, and **adds** new keys.

```
 {"animal": "owl"}  -->  RunnablePassthrough.assign(fact=fact_chain)  -->  {"animal": "owl", "fact": "Owls can..."}
                                                                             (original kept, new key added)
```

You will use this in every RAG chain on Day 6, so it is worth getting comfortable with now.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

print(RunnablePassthrough().invoke({"topic": "AI"}))      # on its own, it just returns the input

In [ ]:
add_length = RunnablePassthrough.assign(topic_length=lambda x: len(x["topic"]))
print(add_length.invoke({"topic": "Generative AI"}))

In [ ]:
keep_input_chain = RunnablePassthrough.assign(fact=fact_chain)
keep_input_chain.invoke({"animal": "owl"})

## 5. Running steps in parallel

If two steps do not depend on each other, why wait for one before starting the other? `RunnableParallel` sends the **same input** to several Runnables at once and collects the results in a dictionary.

```
                          +-->  joke_chain  --> "joke"
  {"topic": "Python"} ----+
                          +-->  poem_chain  --> "poem"
                                       |
                                       v
                       {"joke": "...", "poem": "..."}
```

In [ ]:
from langchain_core.runnables import RunnableParallel

joke_chain = ChatPromptTemplate.from_template("Tell a one-line joke about {topic}.") | llm | parser
poem_chain = ChatPromptTemplate.from_template("Write a 2-line poem about {topic}.") | llm | parser

both = RunnableParallel(joke=joke_chain, poem=poem_chain)

result = both.invoke({"topic": "Python programming"})
print("JOKE:", result["joke"])
print("POEM:", result["poem"])

In [ ]:
both.get_graph().print_ascii()       # notice the two branches side by side

A plain Python dictionary inside a chain is treated as `RunnableParallel` too. This is the style you will see most often in real code:

In [ ]:
combined = {"joke": joke_chain, "poem": poem_chain} | RunnableLambda(lambda d: f"{d['joke']}\n---\n{d['poem']}")
print(combined.invoke({"topic": "coffee"}))

### itemgetter - pick one value out of the input

When your input dict has keys with different names than the prompt expects, `itemgetter` maps them. `itemgetter("sentence")` is just a short way to write `lambda d: d["sentence"]`.

In [ ]:
from operator import itemgetter

translate_chain = (
    {"text": itemgetter("sentence"), "language": itemgetter("lang")}    # rename keys for the prompt
    | ChatPromptTemplate.from_template("Translate to {language}: {text}")
    | llm
    | parser
)

print(translate_chain.invoke({"sentence": "Learning never stops.", "lang": "Hindi"}))

## 6. One LLM call feeding the next

A lot of good results come from breaking a task into steps, the same way a human writer would: first an outline, then the introduction based on that outline.

```
 {"topic"} --> + outline --> {"topic", "outline"} --> + intro --> {"topic", "outline", "intro"}
```

In [ ]:
outline_chain = ChatPromptTemplate.from_template(
    "Create a 4-point blog outline about {topic}. Return only the numbered points."
) | llm | parser

intro_chain = ChatPromptTemplate.from_template(
    "Topic: {topic}\nOutline:\n{outline}\n\nWrite a 3-sentence blog introduction."
) | llm | parser

In [ ]:
blog_chain = (
    RunnablePassthrough.assign(outline=outline_chain)     # adds "outline"
    | RunnablePassthrough.assign(intro=intro_chain)       # uses topic + outline, adds "intro"
)

blog = blog_chain.invoke({"topic": "Why every developer should learn LangChain"})
print("OUTLINE:\n", blog["outline"])
print("\nINTRO:\n", blog["intro"])

A nice side effect: because we keep every intermediate result, debugging is easy. If the intro is bad, you can see whether the outline was already bad.

## 7. Routing - send different questions to different chains

Think of a hospital reception. You describe your problem, and the receptionist sends you to the right department. Let's build that for a support bot: billing questions go to a billing expert prompt, technical questions to a technical one.

```
                                         +-->  billing_chain
 question --> classifier --> category ---+-->  tech_chain
                                         +-->  general_chain
```

First, the receptionist. We use structured output from Day 2 so the category is always one of three exact values.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Route(BaseModel):
    category: Literal["billing", "technical", "general"] = Field(description="Type of customer question")

classifier = (
    ChatPromptTemplate.from_template("Classify this customer question: {question}")
    | llm.with_structured_output(Route)
)

classifier.invoke({"question": "Why was my credit card charged twice?"})

Next, the departments - one small chain each:

In [ ]:
billing_chain = ChatPromptTemplate.from_template(
    "You are a billing specialist. Answer politely in 2 sentences: {question}") | llm | parser

tech_chain = ChatPromptTemplate.from_template(
    "You are a senior support engineer. Give 2 troubleshooting steps: {question}") | llm | parser

general_chain = ChatPromptTemplate.from_template(
    "You are a friendly assistant. Answer in 2 sentences: {question}") | llm | parser

Now connect them. The cleanest way is a normal Python function that **returns the chain to use**. When a function inside a chain returns a Runnable, LangChain runs that Runnable with the same input.

In [ ]:
def pick_department(info):
    if info["category"] == "billing":
        return billing_chain
    if info["category"] == "technical":
        return tech_chain
    return general_chain

router = (
    RunnablePassthrough.assign(category=classifier | (lambda r: r.category))   # add the category
    | RunnableLambda(pick_department)                                           # then route
)

In [ ]:
questions = [
    "Why was my credit card charged twice?",
    "The app crashes when I upload a file larger than 10 MB.",
    "Who founded your company?",
]

for q in questions:
    print("Q:", q)
    print("A:", router.invoke({"question": q}), "\n")

You will also see `RunnableBranch` in older tutorials. It does the same job with `(condition, chain)` pairs. Here it is for comparison - most people find the function version easier to read.

In [ ]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: x["category"] == "billing", billing_chain),
    (lambda x: x["category"] == "technical", tech_chain),
    general_chain,                                           # the default if nothing matches
)

router_v2 = RunnablePassthrough.assign(category=classifier | (lambda r: r.category)) | branch
print(router_v2.invoke({"question": "How do I reset my password?"}))

## 8. When things go wrong: fallbacks and retries

In production, APIs fail: rate limits, timeouts, a model being temporarily unavailable. Two tools help:

- `with_retry()` - try the **same** step again (good for temporary hiccups)
- `with_fallbacks([...])` - if it still fails, try a **different** step (a backup model, a simpler prompt)

```
 primary  --X fails-->  fallback  -->  answer
```

Let's prove fallbacks work by deliberately using a model name that does not exist.

In [ ]:
broken_llm = ChatOpenAI(model="this-model-does-not-exist", max_retries=0)    # this will fail
backup_llm = ChatOpenAI(model=MODEL_NAME)                                     # this works

safe_llm = broken_llm.with_fallbacks([backup_llm])

print(safe_llm.invoke("Say 'fallback worked' and nothing else.").content)

And a retry demo. This function fails on purpose the first two times, like a shaky network.

In [ ]:
attempts = {"count": 0}

def shaky_step(x):
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ValueError(f"Temporary failure on attempt {attempts['count']}")
    return f"Worked on attempt {attempts['count']}"

reliable = RunnableLambda(shaky_step).with_retry(stop_after_attempt=3, wait_exponential_jitter=False)
print(reliable.invoke("hello"))

## 9. Controlling a run with `config`

Every `invoke` and `batch` accepts a `config` dictionary. The one you will use most is `max_concurrency` - it limits how many requests `batch()` sends at once, which keeps you under your API rate limit.

| Key | What it does |
|-----|--------------|
| `max_concurrency` | how many parallel calls `batch()` may make |
| `tags`, `metadata` | labels attached to the run, visible in tracing tools like LangSmith |
| `run_name` | a readable name for the run |

In [ ]:
animals = ["cat", "dog", "horse", "zebra", "lion", "tiger", "bear", "fox", "wolf", "panda"]

facts = fact_chain.batch(
    [{"animal": a} for a in animals],
    config={"max_concurrency": 3, "tags": ["day3"], "run_name": "animal_facts"},
)
for a, f in list(zip(animals, facts))[:3]:
    print(f"{a}: {f}")

**Pause and predict:** if we run the same 10 calls with `max_concurrency` set to 1, 2, 5 and 10, what shape will the time curve have? A straight line down? A curve that flattens out?

In [ ]:
import time
import matplotlib.pyplot as plt

levels, seconds = [1, 2, 5, 10], []
for level in levels:
    start = time.time()
    fact_chain.batch([{"animal": a} for a in animals], config={"max_concurrency": level})
    seconds.append(time.time() - start)

plt.figure(figsize=(5, 3))
plt.plot(levels, seconds, marker="o")
plt.xlabel("max_concurrency")
plt.ylabel("Seconds for 10 calls")
plt.title("More parallel calls, less waiting")
plt.grid(alpha=0.3)
plt.show()

<details>
<summary>Click to see the explanation</summary>

Usually the time drops sharply at first and then flattens. Going from 1 to 2 nearly halves the time, but going from 5 to 10 helps less, because now you are limited by the slowest single request. In practice, pick a value that keeps you under your rate limit.
</details>

## Mini project: a content pipeline that reviews its own work

A marketing team asks: "for any product, give us a tagline, the target audience, a LinkedIn post, and a quick quality check of that post." Let's build it using everything from today.

```
                      +-->  tagline  --+
  {"product"} --------+                +-->  post  -->  review (structured)
                      +-->  audience --+
        (parallel)                       (sequential)
```

In [ ]:
tagline_chain = ChatPromptTemplate.from_template(
    "Write a catchy tagline (max 8 words) for: {product}") | llm | parser

audience_chain = ChatPromptTemplate.from_template(
    "Describe the ideal target audience for {product} in one sentence.") | llm | parser

post_chain = ChatPromptTemplate.from_template(
    "Product: {product}\nTagline: {tagline}\nAudience: {audience}\n"
    "Write a LinkedIn post of at most 60 words. No hashtags."
) | llm | parser

class PostReview(BaseModel):
    clarity: int = Field(description="Score from 1 to 10")
    persuasiveness: int = Field(description="Score from 1 to 10")
    improvement_tip: str = Field(description="One short, specific tip")

review_chain = ChatPromptTemplate.from_template(
    "Review this social media post honestly:\n{post}") | llm.with_structured_output(PostReview)

In [ ]:
content_pipeline = (
    RunnablePassthrough.assign(tagline=tagline_chain, audience=audience_chain)   # these two run in parallel
    | RunnablePassthrough.assign(post=post_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

content_pipeline.get_graph().print_ascii()

In [ ]:
out = content_pipeline.invoke({"product": "CloudDesk - an AI-powered helpdesk for small businesses"})

print("TAGLINE :", out["tagline"])
print("AUDIENCE:", out["audience"])
print("POST    :", out["post"])
print("TIP     :", out["review"].improvement_tip)

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["clarity", "persuasiveness"], [out["review"].clarity, out["review"].persuasiveness],
        color=["#4C72B0", "#55A868"])
plt.ylim(0, 10)
plt.title("How the model rated its own post")
plt.show()

### Your turn

Add one more step to `content_pipeline` that rewrites the post using `improvement_tip`. Hint: one more `RunnablePassthrough.assign(improved_post=...)` with a prompt that uses `{post}` and `{review}`.

In [ ]:
# Write your improved pipeline here

## Common mistakes on Day 3

| What you see | What usually went wrong |
|--------------|-------------------------|
| `KeyError` in the middle of a chain | A step expected a key that an earlier step removed. Use `RunnablePassthrough.assign` to keep it. |
| `TypeError: Expected mapping type` | A prompt received a string instead of a dict. Wrap the value: `{"question": RunnablePassthrough()}`. |
| `print_ascii` fails | Install `grandalf` (it is in requirements.txt) |
| Rate limit errors during `batch()` | Lower `max_concurrency` in the config |

## Quick recap

- Everything is a Runnable, so everything has `invoke`, `batch` and `stream`.
- `|` connects steps. A dict becomes parallel steps. A function becomes a `RunnableLambda`.
- `RunnablePassthrough.assign()` keeps the input and adds new keys.
- Route with a function that returns a chain.
- `with_fallbacks` and `with_retry` make chains survive failures.

## Practice before Day 4

1. Build `text -> summary -> 3 quiz questions about the summary`.
2. Add a fourth department, `"sales"`, to the router.
3. Use a parallel step to summarise a paragraph in English, Hindi and Marathi at the same time.
4. Write a fallback where the backup uses a simpler prompt, not just a different model.

**Tomorrow:** remember yesterday's stateless model? We will give our chatbot a memory - and learn what to do when that memory gets too long.